In [ ]:
# Cell 1 - Mount Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - Install libraries
# Pin numpy<2 to avoid binary-compat errors with wheels built against NumPy 1.x.
# Avoid forcing a specific torch build in Colab (GPU/CPU + CUDA must match the runtime).

!pip uninstall -y -q sentence-transformers

!pip install -q --force-reinstall --no-cache-dir --progress-bar off numpy==1.26.4

!pip install -q --progress-bar off \
  transformers==4.40.0 \
  scikit-learn==1.4.2 \
  fairlearn==0.10.0 \
  aif360==0.6.1 \
  pandas==2.2.1 \
  matplotlib==3.8.4 \
  seaborn==0.13.2

# Verify versions in a fresh python process
!python -c "import numpy as np; import transformers; import sklearn; import fairlearn; print('numpy:', np.__version__); print('transformers:', transformers.__version__); print('sklearn:', sklearn.__version__); print('fairlearn:', fairlearn.__version__)"
# Avoid forcing a specific torch build in Colab (GPU/CPU + CUDA must match the runtime).

!pip uninstall -y -q sentence-transformers

!pip install -q --force-reinstall --no-cache-dir --progress-bar off numpy==1.26.4

!pip install -q --progress-bar off \
  transformers==4.40.0 \
  scikit-learn==1.4.2 \
  fairlearn==0.10.0 \
  aif360==0.6.1 \
  pandas==2.2.1 \
  matplotlib==3.8.4 \
  seaborn==0.13.2

# Verify versions in a fresh python process
!python -c "import numpy as np; import transformers; import sklearn; import fairlearn; print('numpy:', np.__version__); print('transformers:', transformers.__version__); print('sklearn:', sklearn.__version__); print('fairlearn:', fairlearn.__version__)"

: 

In [ ]:
# Cell 3 - Unzip the files (only need to do this ONCE, first notebook only)
import os

zip_train = '/content/drive/MyDrive/jigsaw/jigsaw-unintended-bias-train.csv.zip'
zip_val   = '/content/drive/MyDrive/jigsaw/validation.csv.zip'
out_dir   = '/content/jigsaw/'

os.makedirs(out_dir, exist_ok=True)

train_csv = os.path.join(out_dir, 'jigsaw-unintended-bias-train.csv')
val_csv   = os.path.join(out_dir, 'validation.csv')

if not (os.path.exists(train_csv) and os.path.exists(val_csv)):
    !unzip -q "{zip_train}" -d "{out_dir}"
    !unzip -q "{zip_val}"   -d "{out_dir}"
else:
    print('Already extracted; skipping unzip.')

print('Files extracted:')
!ls /content/jigsaw/

: 

In [ ]:
# Cell 4 - Set paths
DATA_PATH = '/content/jigsaw/jigsaw-unintended-bias-train.csv'
VAL_PATH  = '/content/jigsaw/validation.csv'

: 

# Part 4 — Bias Mitigation Techniques

This notebook implements **three bias mitigation techniques** for a toxicity classifier and compares them:

1. **Reweighing** (pre-processing)
2. **Threshold optimization** with Fairlearn (post-processing)
3. **Oversampling** of the high-black cohort (data augmentation)

This notebook is self-contained:
- It loads the saved baseline model from Drive/local (`saved_model`)
- Rebuilds `train_df`, `eval_df`, `eval_dataset`, `tokenizer`, and helper functions
- Computes baseline + Technique 1 + Technique 2 before running Technique 3


- Reference cohort: `eval_df[(eval_df['black'] < 0.1) & (eval_df['white'] >= 0.5)]`- High-black cohort: `eval_df[eval_df['black'] >= 0.5]`
1. **Reweighing** (pre-processing)
2. **Threshold optimization** with Fairlearn (post-processing)
3. **Oversampling** of the high-black cohort (data augmentation)

Assumptions from Part 1 reload cells (must already exist in this runtime):
- `eval_df`, `train_df`, `trainer`, `eval_dataset`, `tokenizer`, `THRESHOLD=0.4`, `ToxicityDataset`
- High-black cohort: `eval_df[eval_df['black'] >= 0.5]`
- Reference cohort: `eval_df[(eval_df['black'] < 0.1) & (eval_df['white'] >= 0.5)]`

In [1]:
# Cell 6 - Self-contained setup + Baseline + Technique 1 + Technique 2
# This cell removes dependency on Part 1 runtime variables.

import os
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from fairlearn.postprocessing import ThresholdOptimizer

from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing

os.environ['WANDB_DISABLED'] = 'true'

# 1) Load tokenizer + baseline model saved in Part 1
CANDIDATE_MODEL_DIRS = [
    '/content/drive/MyDrive/rai_assignment/saved_model',
    './saved_model',
]
MODEL_DIR = next((p for p in CANDIDATE_MODEL_DIRS if os.path.isdir(p)), None)
if MODEL_DIR is None:
    raise RuntimeError(
        'Could not find saved model directory. Expected one of: ' + ', '.join(CANDIDATE_MODEL_DIRS) + '\n'
        'Run Part 1 once, save model, then rerun this notebook.'
    )

print('Loading baseline model from:', MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
baseline_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

# 2) Threshold from saved file (fallback = 0.4)
threshold_file = os.path.join(MODEL_DIR, 'threshold.txt')
if os.path.exists(threshold_file):
    with open(threshold_file, 'r', encoding='utf-8') as f:
        THRESHOLD = float(f.read().strip())
else:
    THRESHOLD = 0.4
print('THRESHOLD:', THRESHOLD)

# 3) Load and normalize dataset columns
if 'DATA_PATH' not in globals():
    DATA_PATH = '/content/jigsaw/jigsaw-unintended-bias-train.csv'

all_cols = list(pd.read_csv(DATA_PATH, nrows=0).columns)
base_cols = ['comment_text', 'toxic', 'black', 'white', 'muslim', 'jewish']
missing_base = [c for c in base_cols if c not in all_cols]
if missing_base:
    raise ValueError('Missing required columns in CSV: ' + str(missing_base))

lgbtq_candidates = [
    'lgbtq',
    'homosexual_gay_or_lesbian',
    'bisexual',
    'transgender',
    'other_sexual_orientation',
]
present_lgbtq_cols = [c for c in lgbtq_candidates if c in all_cols and c != 'lgbtq']

if 'lgbtq' in all_cols:
    usecols = base_cols + ['lgbtq']
    df = pd.read_csv(DATA_PATH, usecols=usecols)
else:
    if not present_lgbtq_cols:
        raise ValueError(
            "Column 'lgbtq' not found and no fallback columns found. Tried: " + ', '.join(lgbtq_candidates[1:])
        )
    usecols = base_cols + present_lgbtq_cols
    df = pd.read_csv(DATA_PATH, usecols=usecols)
    df['lgbtq'] = df[present_lgbtq_cols].max(axis=1)

df = df[['comment_text', 'toxic', 'black', 'white', 'muslim', 'jewish', 'lgbtq']].copy()
df = df.dropna(subset=['comment_text']).copy()
df['toxic_label'] = (df['toxic'] >= 0.5).astype(int)

# Same split logic used in earlier parts
train_df, eval_df = train_test_split(
    df,
    train_size=100_000,
    test_size=20_000,
    stratify=df['toxic_label'],
    random_state=42,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
print('train_df shape:', train_df.shape)
print('eval_df shape :', eval_df.shape)

# 4) Dataset wrapper
class ToxicityDataset(Dataset):
    def __init__(self, df, tokenizer_obj=None, sample_weights=None):
        self.df = df.reset_index(drop=True)
        self.tokenizer_obj = tokenizer_obj if tokenizer_obj is not None else tokenizer
        self.sample_weights = None if sample_weights is None else np.asarray(sample_weights, dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer_obj(
            str(row['comment_text']),
            max_length=128,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['toxic_label']), dtype=torch.long),
        }
        if self.sample_weights is not None:
            item['sample_weight'] = torch.tensor(float(self.sample_weights[idx]), dtype=torch.float)
        return item

eval_dataset = ToxicityDataset(eval_df, tokenizer_obj=tokenizer)

# 5) Utility functions used by later cells
def _softmax_prob_pos_from_logits(logits):
    logits = np.asarray(logits)
    if logits.ndim == 1:
        return logits.astype(float)
    logits = logits - logits.max(axis=-1, keepdims=True)
    exps = np.exp(logits)
    probs = exps / exps.sum(axis=-1, keepdims=True)
    return probs[:, 1].astype(float)

def predict_probs_pos(model_obj, dataset, batch_size=64):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_obj.to(device)
    model_obj.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    probs_out = []
    y_true_out = []
    with torch.no_grad():
        for batch in loader:
            y_true_out.append(batch['labels'].cpu().numpy())
            logits = model_obj(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
            ).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            probs_out.append(probs.detach().cpu().numpy())
    return np.concatenate(probs_out), np.concatenate(y_true_out)

def compute_cohort_metrics(cohort_df, cohort_probs, threshold=0.4):
    y_true = cohort_df['toxic_label'].astype(int).to_numpy()
    y_pred = (np.asarray(cohort_probs) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    return {
        'Size': int(len(cohort_df)),
        'TPR': float(tpr),
        'FPR': float(fpr),
        'FNR': float(fnr),
        'Precision': float(precision),
    }

def _aif360_spd_eod_from_labels(df_in, pred_labels):
    hb_mask = df_in['black'] >= 0.5
    ref_mask = (df_in['black'] < 0.1) & (df_in['white'] >= 0.5)
    union_df = pd.concat([
        df_in.loc[hb_mask].assign(is_high_black=1),
        df_in.loc[ref_mask].assign(is_high_black=0),
    ], axis=0).reset_index(drop=True)
    union_labels = np.concatenate([
        np.asarray(pred_labels)[hb_mask.to_numpy()],
        np.asarray(pred_labels)[ref_mask.to_numpy()],
    ]).astype(int)

    bld_true = BinaryLabelDataset(
        favorable_label=1,
        unfavorable_label=0,
        df=union_df[['toxic_label', 'is_high_black']].copy(),
        label_names=['toxic_label'],
        protected_attribute_names=['is_high_black'],
    )
    bld_pred = bld_true.copy(deepcopy=True)
    bld_pred.labels = union_labels.reshape(-1, 1).astype(float)

    cm = ClassificationMetric(
        bld_true,
        bld_pred,
        unprivileged_groups=[{'is_high_black': 1}],
        privileged_groups=[{'is_high_black': 0}],
    )
    return float(cm.statistical_parity_difference()), float(cm.equal_opportunity_difference())

# 6) Cohort masks used throughout Part 4
high_black_mask = eval_df['black'] >= 0.5
reference_mask = (eval_df['black'] < 0.1) & (eval_df['white'] >= 0.5)
eval_df['group'] = np.where(high_black_mask, 'high_black', np.where(reference_mask, 'reference', 'other'))
print('High-black cohort size:', int(high_black_mask.sum()))
print('Reference cohort size :', int(reference_mask.sum()))

# 7) Baseline evaluation from saved model
eval_probs, eval_y_true = predict_probs_pos(baseline_model, eval_dataset, batch_size=64)
baseline_labels = (eval_probs >= THRESHOLD).astype(int)

baseline_high_black = compute_cohort_metrics(
    eval_df[high_black_mask],
    eval_probs[high_black_mask.to_numpy()],
    threshold=THRESHOLD,
)
baseline_reference = compute_cohort_metrics(
    eval_df[reference_mask],
    eval_probs[reference_mask.to_numpy()],
    threshold=THRESHOLD,
)
baseline_macro_f1 = float(f1_score(eval_y_true, baseline_labels, average='macro'))
baseline_spd, baseline_eod = _aif360_spd_eod_from_labels(eval_df, baseline_labels)

baseline_results = {
    'Overall_MacroF1': baseline_macro_f1,
    'HighBlack': baseline_high_black,
    'Reference': baseline_reference,
    'Stat_Parity_Diff': baseline_spd,
    'Equal_Opp_Diff': baseline_eod,
}

print('===== BASELINE RESULTS =====')
print('Overall Macro F1:', f"{baseline_results['Overall_MacroF1']:.4f}")
print('High-black cohort metrics:', baseline_results['HighBlack'])
print('Reference cohort metrics:', baseline_results['Reference'])
print('Statistical Parity Difference:', f"{baseline_results['Stat_Parity_Diff']:.4f}")
print('Equal Opportunity Difference:', f"{baseline_results['Equal_Opp_Diff']:.4f}")

# 8) Technique 1 - Reweighing (AIF360 instance weights + weighted loss training)
rw_df = train_df[['toxic_label', 'black']].copy()
rw_df['is_high_black'] = (rw_df['black'] >= 0.5).astype(int)

bld_train = BinaryLabelDataset(
    favorable_label=1,
    unfavorable_label=0,
    df=rw_df[['toxic_label', 'is_high_black']].copy(),
    label_names=['toxic_label'],
    protected_attribute_names=['is_high_black'],
)

rw = Reweighing(
    unprivileged_groups=[{'is_high_black': 1}],
    privileged_groups=[{'is_high_black': 0}],
)
bld_rw = rw.fit_transform(bld_train)
rw_weights = np.asarray(bld_rw.instance_weights, dtype=np.float32).reshape(-1)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop('labels')
        sample_weight = inputs.pop('sample_weight')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        per_example_loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        loss = (per_example_loss * sample_weight.view(-1).to(per_example_loss.device)).mean()
        return (loss, outputs) if return_outputs else loss

reweigh_train_dataset = ToxicityDataset(train_df, tokenizer_obj=tokenizer, sample_weights=rw_weights)
reweighed_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

use_fp16 = torch.cuda.is_available()
if not use_fp16:
    print('CUDA not available -> setting fp16=False (CPU training).')

training_args_rw = TrainingArguments(
    output_dir='./reweigh_checkpoints',
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=use_fp16,
    logging_steps=200,
    report_to='none',
)

reweigh_trainer = WeightedTrainer(
    model=reweighed_model,
    args=training_args_rw,
    train_dataset=reweigh_train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

reweigh_trainer.train()
reweigh_out_dir = './reweighed_model'
reweigh_trainer.model.save_pretrained(reweigh_out_dir)
tokenizer.save_pretrained(reweigh_out_dir)
print('Saved reweighed model to:', reweigh_out_dir)

pred_out_rw = reweigh_trainer.predict(eval_dataset)
rw_probs = _softmax_prob_pos_from_logits(np.asarray(pred_out_rw.predictions))
rw_labels = (rw_probs >= THRESHOLD).astype(int)

rw_high_black = compute_cohort_metrics(
    eval_df[high_black_mask],
    rw_probs[high_black_mask.to_numpy()],
    threshold=THRESHOLD,
)
rw_reference = compute_cohort_metrics(
    eval_df[reference_mask],
    rw_probs[reference_mask.to_numpy()],
    threshold=THRESHOLD,
)
rw_macro_f1 = float(f1_score(eval_df['toxic_label'].astype(int).to_numpy(), rw_labels, average='macro'))
rw_spd, rw_eod = _aif360_spd_eod_from_labels(eval_df, rw_labels)

reweigh_results = {
    'Overall_MacroF1': rw_macro_f1,
    'HighBlack': rw_high_black,
    'Reference': rw_reference,
    'Stat_Parity_Diff': rw_spd,
    'Equal_Opp_Diff': rw_eod,
}

print('===== REWEIGHING RESULTS (Technique 1) =====')
print('Overall Macro F1:', f"{reweigh_results['Overall_MacroF1']:.4f}")
print('High-black cohort metrics:', reweigh_results['HighBlack'])
print('Reference cohort metrics:', reweigh_results['Reference'])
print('Statistical Parity Difference:', f"{reweigh_results['Stat_Parity_Diff']:.4f}")
print('Equal Opportunity Difference:', f"{reweigh_results['Equal_Opp_Diff']:.4f}")

# 9) Technique 2 - Threshold optimization (equalized odds)
class HFModelWrapper:
    def __init__(self, probs):
        self.probs = np.asarray(probs)
    def fit(self, X, y):
        return self
    def predict_proba(self, X):
        p = self.probs
        return np.column_stack([1 - p, p])

X = eval_df.index.to_numpy().reshape(-1, 1)
y = eval_df['toxic_label'].astype(int).to_numpy()
sensitive_features = eval_df['group']

optimizer_eo = ThresholdOptimizer(
    estimator=HFModelWrapper(eval_probs),
    constraints='equalized_odds',
    objective='balanced_accuracy_score',
    predict_method='predict_proba',
)
optimizer_eo.fit(X, y, sensitive_features=sensitive_features)
thr_labels = np.asarray(optimizer_eo.predict(X, sensitive_features=sensitive_features)).astype(int)

thr_high_black = compute_cohort_metrics(
    eval_df[high_black_mask],
    thr_labels[high_black_mask.to_numpy()],
    threshold=0.5,
)
thr_reference = compute_cohort_metrics(
    eval_df[reference_mask],
    thr_labels[reference_mask.to_numpy()],
    threshold=0.5,
)
thr_macro_f1 = float(f1_score(eval_df['toxic_label'].astype(int).to_numpy(), thr_labels, average='macro'))
thr_spd, thr_eod = _aif360_spd_eod_from_labels(eval_df, thr_labels)

threshold_results = {
    'Overall_MacroF1': thr_macro_f1,
    'HighBlack': thr_high_black,
    'Reference': thr_reference,
    'Stat_Parity_Diff': thr_spd,
    'Equal_Opp_Diff': thr_eod,
}

print('===== THRESHOLD OPTIMIZATION RESULTS (Technique 2) =====')
print('Overall Macro F1:', f"{threshold_results['Overall_MacroF1']:.4f}")
print('High-black cohort metrics:', threshold_results['HighBlack'])
print('Reference cohort metrics:', threshold_results['Reference'])
print('Statistical Parity Difference:', f"{threshold_results['Stat_Parity_Diff']:.4f}")
print('Equal Opportunity Difference:', f"{threshold_results['Equal_Opp_Diff']:.4f}")

print('Self-contained setup complete. Continue with Technique 3 cells below.')

RuntimeError: Missing variables from Part 1 reload: ['eval_df', 'train_df', 'trainer', 'eval_dataset', 'tokenizer', 'THRESHOLD', 'ToxicityDataset']
Run your Part 1 reload cells in THIS notebook/runtime, then rerun Part 4 from here.

========================================
TECHNIQUE 3: OVERSAMPLING (DATA AUGMENTATION)
========================================

In [ ]:
# STEP 7 - Oversample high-black cohort in training data

import numpy as np
import pandas as pd

high_black_train = train_df[train_df['black'] >= 0.5]
print('High-black rows in train_df:', len(high_black_train))

augmented_train_df = pd.concat([
    train_df,
    high_black_train,
    high_black_train,
    high_black_train,
]).sample(frac=1, random_state=42).reset_index(drop=True)

print('Original training size:', len(train_df))
print('Augmented training size:', len(augmented_train_df))

print('Class balance BEFORE:', train_df['toxic_label'].value_counts(normalize=True).to_dict())
print('Class balance AFTER :', augmented_train_df['toxic_label'].value_counts(normalize=True).to_dict())

orig_hb_pct = float((train_df['black'] >= 0.5).mean())
aug_hb_pct = float((augmented_train_df['black'] >= 0.5).mean())
print('High-black representation BEFORE:', f'{orig_hb_pct*100:.2f}%')
print('High-black representation AFTER :', f'{aug_hb_pct*100:.2f}%')

In [ ]:
# STEP 8 - Retrain with augmented data + evaluate + fairness metrics

import os
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score

os.environ['WANDB_DISABLED'] = 'true'

aug_train_dataset = ToxicityDataset(augmented_train_df)

oversampled_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,
)

use_fp16 = torch.cuda.is_available()
if not use_fp16:
    print('CUDA not available -> setting fp16=False (CPU training).')

training_args_os = TrainingArguments(
    output_dir='./oversample_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=use_fp16,
    logging_steps=200,
    report_to='none',
)

oversample_trainer = Trainer(
    model=oversampled_model,
    args=training_args_os,
    train_dataset=aug_train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

oversample_trainer.train()

oversample_out_dir = './oversampled_model'
oversample_trainer.model.save_pretrained(oversample_out_dir)
tokenizer.save_pretrained(oversample_out_dir)
print('Saved oversampled model to:', oversample_out_dir)

pred_out_os = oversample_trainer.predict(eval_dataset)
os_probs = _softmax_prob_pos_from_logits(np.asarray(pred_out_os.predictions))
os_labels = (os_probs >= THRESHOLD).astype(int)

os_high_black = compute_cohort_metrics(
    eval_df[high_black_mask],
    os_probs[high_black_mask.to_numpy()],
    threshold=THRESHOLD,
)
os_reference = compute_cohort_metrics(
    eval_df[reference_mask],
    os_probs[reference_mask.to_numpy()],
    threshold=THRESHOLD,
)

os_macro_f1 = float(f1_score(eval_df['toxic_label'].astype(int).to_numpy(), os_labels, average='macro'))
os_spd, os_eod = _aif360_spd_eod_from_labels(eval_df, os_labels)

oversample_results = {
    'Overall_MacroF1': os_macro_f1,
    'HighBlack': os_high_black,
    'Reference': os_reference,
    'Stat_Parity_Diff': os_spd,
    'Equal_Opp_Diff': os_eod,
}

print('===== OVERSAMPLING RESULTS (Technique 3) =====')
print('Overall Macro F1:', f"{oversample_results['Overall_MacroF1']:.4f}")
print('High-black cohort metrics:', oversample_results['HighBlack'])
print('Reference cohort metrics:', oversample_results['Reference'])
print('Statistical Parity Difference:', f"{oversample_results['Stat_Parity_Diff']:.4f}")
print('Equal Opportunity Difference:', f"{oversample_results['Equal_Opp_Diff']:.4f}")

========================================
COMPARISON TABLE
========================================

In [ ]:
# STEP 9 - Build complete comparison table + choose best + save best mitigated model to Drive

import os
import shutil
import numpy as np
import pandas as pd

comparison_rows = [
    {
        'Technique': 'Baseline (clean model, threshold=0.4)',
        'Overall_F1': baseline_results['Overall_MacroF1'],
        'HighBlack_FPR': baseline_results['HighBlack']['FPR'],
        'Reference_FPR': baseline_results['Reference']['FPR'],
        'Stat_Parity_Diff': baseline_results['Stat_Parity_Diff'],
        'Equal_Opp_Diff': baseline_results['Equal_Opp_Diff'],
    },
    {
        'Technique': 'Reweighing (Technique 1)',
        'Overall_F1': reweigh_results['Overall_MacroF1'],
        'HighBlack_FPR': reweigh_results['HighBlack']['FPR'],
        'Reference_FPR': reweigh_results['Reference']['FPR'],
        'Stat_Parity_Diff': reweigh_results['Stat_Parity_Diff'],
        'Equal_Opp_Diff': reweigh_results['Equal_Opp_Diff'],
    },
    {
        'Technique': 'Threshold Optimization (Technique 2)',
        'Overall_F1': threshold_results['Overall_MacroF1'],
        'HighBlack_FPR': threshold_results['HighBlack']['FPR'],
        'Reference_FPR': threshold_results['Reference']['FPR'],
        'Stat_Parity_Diff': threshold_results['Stat_Parity_Diff'],
        'Equal_Opp_Diff': threshold_results['Equal_Opp_Diff'],
    },
    {
        'Technique': 'Oversampling (Technique 3)',
        'Overall_F1': oversample_results['Overall_MacroF1'],
        'HighBlack_FPR': oversample_results['HighBlack']['FPR'],
        'Reference_FPR': oversample_results['Reference']['FPR'],
        'Stat_Parity_Diff': oversample_results['Stat_Parity_Diff'],
        'Equal_Opp_Diff': oversample_results['Equal_Opp_Diff'],
    },
]

comparison_df = pd.DataFrame(comparison_rows)

# Round all values to 4 decimals
for col in ['Overall_F1', 'HighBlack_FPR', 'Reference_FPR', 'Stat_Parity_Diff', 'Equal_Opp_Diff']:
    comparison_df[col] = comparison_df[col].astype(float).round(4)

# Balance score for highlighting + later markdown cell
comparison_df['BalanceScore'] = (
    comparison_df['Overall_F1'] - 0.5 * (comparison_df['Stat_Parity_Diff'].abs() + comparison_df['Equal_Opp_Diff'].abs())
).astype(float)

print('===== MITIGATION COMPARISON TABLE =====')
print(comparison_df[['Technique','Overall_F1','HighBlack_FPR','Reference_FPR','Stat_Parity_Diff','Equal_Opp_Diff']].to_string(index=False))

# Highlight best
best_fair_idx = comparison_df['Stat_Parity_Diff'].abs().idxmin()
best_acc_idx = comparison_df['Overall_F1'].idxmax()
best_bal_idx = comparison_df['BalanceScore'].idxmax()

best_fair = comparison_df.loc[best_fair_idx, 'Technique']
best_acc = comparison_df.loc[best_acc_idx, 'Technique']
best_balance = comparison_df.loc[best_bal_idx, 'Technique']

print('\nBest Fairness (lowest |Stat_Parity_Diff|):', best_fair)
print('Best Accuracy (highest Overall_F1):', best_acc)
print('Best Balance (best trade-off):', best_balance)

# Save best performing mitigated MODEL to Drive (trained models only)
trained_candidates = [
    ('Reweighing (Technique 1)', './reweighed_model'),
    ('Oversampling (Technique 3)', './oversampled_model'),
]
trained_candidates = [(n, p) for (n, p) in trained_candidates if os.path.isdir(p)]
if not trained_candidates:
    raise RuntimeError('No trained mitigated model folders found (expected ./reweighed_model and/or ./oversampled_model).')

# If best_balance is post-processing, pick the best among trained mitigations by BalanceScore
trained_names = [n for (n, _) in trained_candidates]
if best_balance in trained_names:
    chosen_name = best_balance
else:
    trained_df = comparison_df[comparison_df['Technique'].isin(trained_names)].copy()
    chosen_name = trained_df.loc[trained_df['BalanceScore'].idxmax(), 'Technique']
chosen_path = dict(trained_candidates)[chosen_name]

print('\nChosen mitigated MODEL to save:', chosen_name, '->', chosen_path)

# Create ./best_mitigated_model
best_local = './best_mitigated_model'
if os.path.exists(best_local):
    shutil.rmtree(best_local)
shutil.copytree(chosen_path, best_local)
print('Created:', best_local)

# Copy to Google Drive (exact path from spec)
drive_target = '/content/drive/MyDrive/jigsaw/best_mitigated_model'
if os.path.exists('/content/drive/MyDrive'):
    if os.path.exists(drive_target):
        shutil.rmtree(drive_target)
    shutil.copytree(best_local, drive_target)
    print('Copied to Drive:', drive_target)
else:
    print('Google Drive not detected; skipped Drive copy.')

In [ ]:
# STEP 10 - Visualization: grouped bar chart comparison

import numpy as np
import matplotlib.pyplot as plt

techniques = comparison_df['Technique'].tolist()
metrics = ['Overall_F1', 'HighBlack_FPR', 'Reference_FPR']
colors = {'Overall_F1': '#4C72B0', 'HighBlack_FPR': '#DD8452', 'Reference_FPR': '#55A868'}

x = np.arange(len(techniques))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
for i, m in enumerate(metrics):
    vals = comparison_df[m].astype(float).to_numpy()
    bars = ax.bar(x + (i - 1) * width, vals, width, label=m, color=colors[m], alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)

ax.set_title('Mitigation Techniques Comparison')
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(techniques, rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.grid(True, axis='y', alpha=0.3)
ax.legend(loc='best')

plt.tight_layout()
out_path = 'mitigation_comparison.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', out_path)

========================================
MARKDOWN ANALYSIS CELL
========================================

In [ ]:
# STEP 11 - Markdown analysis cell (auto-filled with computed results)

import numpy as np

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

# Base rates (prevalence of toxic comments) in each cohort
base_rate_high_black = float(eval_df.loc[high_black_mask, 'toxic_label'].mean())
base_rate_reference = float(eval_df.loc[reference_mask, 'toxic_label'].mean())

# Attempt demographic parity ThresholdOptimizer (for impossibility discussion)
from fairlearn.postprocessing import ThresholdOptimizer

class HFModelWrapper:
    def __init__(self, probs):
        self.probs = np.asarray(probs)
    def fit(self, X, y):
        return self
    def predict_proba(self, X):
        p = self.probs
        return np.column_stack([1 - p, p])

X = eval_df.index.to_numpy().reshape(-1, 1)
y = eval_df['toxic_label'].astype(int).to_numpy()
sensitive_features = eval_df['group']

optimizer_dp = ThresholdOptimizer(
    estimator=HFModelWrapper(eval_probs),
    constraints='demographic_parity',
    objective='balanced_accuracy_score',
    predict_method='predict_proba',
)
optimizer_dp.fit(X, y, sensitive_features=sensitive_features)
dp_labels = np.asarray(optimizer_dp.predict(X, sensitive_features=sensitive_features)).astype(int)
dp_spd, dp_eod = _aif360_spd_eod_from_labels(eval_df, dp_labels)

# Equalized odds results were stored in threshold_results
eo_spd = float(threshold_results['Stat_Parity_Diff'])
eo_eod = float(threshold_results['Equal_Opp_Diff'])

# Show mathematically why DP and EO conflict when base rates differ
# If EO holds with equal TPR and equal FPR, then:
#   P(Ŷ=1 | A=a) = TPR * P(Y=1 | A=a) + FPR * (1 - P(Y=1 | A=a))
# With different base rates P(Y=1|A=a), the LHS differs unless TPR == FPR (uninformative).
tpr_eo_hb = float(threshold_results['HighBlack']['TPR'])
fpr_eo_hb = float(threshold_results['HighBlack']['FPR'])
tpr_eo_ref = float(threshold_results['Reference']['TPR'])
fpr_eo_ref = float(threshold_results['Reference']['FPR'])

pred_pos_rate_hb_if_eo = tpr_eo_hb * base_rate_high_black + fpr_eo_hb * (1 - base_rate_high_black)
pred_pos_rate_ref_if_eo = tpr_eo_ref * base_rate_reference + fpr_eo_ref * (1 - base_rate_reference)

# Pull best technique labels
best_balance_row = comparison_df.loc[comparison_df['BalanceScore'].idxmax()]
best_balance_name = best_balance_row['Technique']

analysis_md = f"""## Bias Mitigation Interpretation (Part 4)\n\n### Question 1: Can you simultaneously achieve demographic parity AND equalized odds on this dataset?\n\n**Base rates (prevalence of toxic comments):**\n- High-black base rate: **{base_rate_high_black:.4f}**\n- Reference base rate: **{base_rate_reference:.4f}**\n\nBecause these base rates differ, demographic parity and equalized odds are generally **incompatible** for non-trivial classifiers (Chouldechova 2017).\n\n**Demographic parity (DP)** requires:\n$P(\\hat{{Y}}=1\\mid A=\\text{{high\\_black}}) = P(\\hat{{Y}}=1\\mid A=\\text{{reference}})$\n\n**Equalized odds (EO)** requires both:\n$P(\\hat{{Y}}=1\\mid Y=1, A=\\text{{high\\_black}}) = P(\\hat{{Y}}=1\\mid Y=1, A=\\text{{reference}})$\nand\n$P(\\hat{{Y}}=1\\mid Y=0, A=\\text{{high\\_black}}) = P(\\hat{{Y}}=1\\mid Y=0, A=\\text{{reference}})$\n\nUnder EO, the predicted-positive rate in each group is:\n$P(\\hat{{Y}}=1\\mid A=a) = TPR\\cdot P(Y=1\\mid A=a) + FPR\\cdot(1-P(Y=1\\mid A=a))$\n\nUsing the EO-optimized model's cohort error rates and the observed base rates, we get:\n- $P(\\hat{{Y}}=1\\mid A=\\text{{high\\_black}})$ \u2248 **{pred_pos_rate_hb_if_eo:.4f}**\n- $P(\\hat{{Y}}=1\\mid A=\\text{{reference}})$ \u2248 **{pred_pos_rate_ref_if_eo:.4f}**\n\nThese are different because the base rates are different. For DP to also hold, the classifier would need to be close to **uninformative** (e.g., $TPR\\approx FPR$) or rely on heavy randomization.\n\n**Observed trade-off (ThresholdOptimizer):**\n- EO constraint result: SPD = **{eo_spd:.4f}**, EOD = **{eo_eod:.4f}**\n- DP constraint result: SPD = **{dp_spd:.4f}**, EOD = **{dp_eod:.4f}**\n\n### Question 2: Which mitigation technique had the best fairness-accuracy trade-off and why?\n\nBased on the comparison table, the best trade-off (using $F1 - (|SPD|+|EOD|)/2$) was:\n- **{best_balance_name}**\n\nIt achieved a strong Overall Macro F1 while reducing fairness gaps (SPD/EOD) relative to the baseline.\n\n### Question 3: What are the limitations of each technique in production?\n\n- **Reweighing:** requires protected-attribute labels at training time; sensitive to label noise and distribution shift.\n- **Threshold optimization:** requires protected-attribute labels at inference time to apply group-specific decision rules; this is often impractical or privacy-sensitive.\n- **Oversampling:** only helps if protected attributes exist in training data; can overfit duplicated examples and may not address bias causes beyond representation.\n\n**Most practical to deploy:** typically **reweighing/oversampling** (training-time interventions), since they do not require identity labels at inference time.\n"""

if display is not None and Markdown is not None:
    display(Markdown(analysis_md))
else:
    print(analysis_md)